# Hierarchical Modules: Controller Plus Sim

In this tutorial, we'll show how you can combine a simulation module `CometMirror` with a controller module that tries to maintain a certain stored energy.

Let's first load up an instance of CometMirror.

In [ ]:
%load_ext autoreload
%autoreload 2


from popsim.simulate import make_time_base
from popsim.simulators.comet_mirror.scenarios.sparc_prd import build_comet_mirror_config

model, state, inputs = build_comet_mirror_config()
dt = 0.01
time_base = make_time_base(t0=0.0, t1=5.0, dt=dt)

## Instantiating a PID Controller.
Now let's instantiate a generic PID controller that will determine the level of auxiliary heating.

In [ ]:
from popsim.modules.pid import PIDController

pid = PIDController(config=PIDController.Config(Kp=10.0, Ki=0.1, Kd=1.0, dt=dt, output_min=0.0, output_max=15.0))

To better understand how to use it, let's take a look at the source code of the controller. We see that once we have determined the controller gains, we need to provide it with the current setpoint, current measurement, and an optional feed-forward and it will return the control signal.

We also see that it needs to track an array of errors to calculate the integral term that we will need to provide. Note the `discrete_time_field()`
field tells the `simulate` function that this state variable is not a time variable, but should be carried over from one time step to the next.

In [ ]:
import inspect

from IPython.display import Markdown, display

display(Markdown(f"```python\n{inspect.getsource(PIDController)}\n```"))

## Coupling the Controller with CometMirror
Okay, now that we have the controller and the CometMirror model, we can couple them together and simulate the system. We do this by defining a new module that owns both a `CometMirror` and a `PIDController` and manages the interface between them.

Note that we still use the `CometMirror.Inputs` as a part of this new modules `Inputs` struct. However, `Inputs.P_aux_MW` is no longer the heating that will be applied; it is rather the feed-forward heating that will be applied; the controller will determine the additional heating to be applied. Mathematically, this is expressed as:

$$P_{aux} = P_{aux,FF} + PID(W_{th, meas}, W_{th,target})$$

You can see the software realization in the code below.

In [ ]:
import chex
import equinox as eqx

from popsim import ModuleBase
from popsim.simulate import SimInput, simulate
from popsim.simulators.comet_mirror.model import CometMirror


@chex.dataclass
class CometMirrorWithICRFControl(ModuleBase):
    @chex.dataclass
    class Config:
        comet_mirror: CometMirror
        pid_controller: PIDController

    @chex.dataclass
    class State:
        cm_state: CometMirror.State
        controller_state: PIDController.State

    @chex.dataclass
    class Inputs:
        cm_inputs: CometMirror.Inputs
        stored_energy_setpoint: float

    @chex.dataclass
    class Output:
        cm_output: dict
        controller_output: PIDController.Output
        aux_data: dict = None

    config: Config

    def __init__(self, config: Config):
        self.config = config

    def __call__(self, state: "State", inputs: "Inputs") -> tuple:
        # Call the PID controller.
        # Note we use provide a feed-forward auxilary heating term to the PID controller in the form of inputs.cm_inputs.P_aux_MW.
        controller_inputs = PIDController.Inputs(
            setpoint=inputs.stored_energy_setpoint, measurement=state.cm_state.stored_energy, feed_forward=inputs.cm_inputs.P_aux_MW
        )
        next_controller_state, controller_output = self.config.pid_controller(state.controller_state, controller_inputs)

        # Apply PID controller aux heating to the comet mirror by overriding the aux heating in the inputs.
        cm_inputs_in = eqx.tree_at(lambda p: p.P_aux_MW, inputs.cm_inputs, controller_output.control)
        cm_state_dot, cm_output = self.config.comet_mirror(state.cm_state, cm_inputs_in)

        # Return the updated state and output.
        state_out = CometMirrorWithICRFControl.State(cm_state=cm_state_dot, controller_state=next_controller_state)
        output = CometMirrorWithICRFControl.Output(
            cm_output=cm_output, controller_output=controller_output, aux_data={"cm_inputs_in": cm_inputs_in}
        )
        return state_out, output


module = CometMirrorWithICRFControl(
    config=CometMirrorWithICRFControl.Config(
        comet_mirror=model,
        pid_controller=pid,
    )
)
initial_state = CometMirrorWithICRFControl.State(cm_state=state, controller_state=PIDController.State())
inputs_control = CometMirrorWithICRFControl.Inputs(cm_inputs=inputs, stored_energy_setpoint=23.0)

dataset = simulate(module=module, sim_inputs=SimInput(time=time_base, initial_state=initial_state, inputs=inputs_control))

In [ ]:
dataset

In [ ]:
from popsim.visualize import visualize_time_series

visualize_vars = [
    "state.cm_state.stored_energy",
    ("output.controller_output.aux_data.P", "output.controller_output.aux_data.I", "output.controller_output.aux_data.D"),
    ("output.controller_output.control", "output.aux_data.cm_inputs_in.P_aux_MW"),
    "output.cm_output.aux_data.P_rad_MW",
    "state.cm_state.hmode_state.hmode",
]
visualize_time_series(dataset, visualize_vars, max_cols=2)